In [5]:
import networkx as nx

%load_ext autoreload
%autoreload 2

from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# Load the GraphML file
g = nx.read_graphml("../data/repo_graph.graphml")

In [7]:
node_id = "utgen/raggraph/walker.py::function::iter_python_files"
node_id = "utgen/raggraph/utils.py::function::normalize_signature"

context = get_node_context(g=g, node_id=node_id)
print(context)

### NODE INFO ###
file::type::name -> utgen/raggraph/utils.py::function::normalize_signature

source:
def normalize_signature(node):
    """Return a Python-like signature including annotations and return type."""
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return ""

    def unparse(x):
        try:
            return ast.unparse(x)
        except Exception:
            return ""

    parts = []

    # Pos-only args (Python 3.8+)
    for a in getattr(node.args, "posonlyargs", []):
        s = a.arg
        if a.annotation:
            s += f": {unparse(a.annotation)}"
        parts.append(s)
    if getattr(node.args, "posonlyargs", []):
        parts.append("/")

    # Regular args
    for a in node.args.args:
        s = a.arg
        if a.annotation:
            s += f": {unparse(a.annotation)}"
        parts.append(s)

    # Vararg
    if node.args.vararg:
        s = "*" + node.args.vararg.arg
        if node.args.vararg.annotation:
            s +

In [8]:
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

# Extract metadata
inputs = {"graph_context": context}

response = test_generator.crew().kickoff(inputs=inputs)

In [9]:
print(response.raw)

{"tests": {"test_normalize_signature_with_basic_function": {"name": "test_normalize_signature_with_basic_function", "imports": "import ast\nimport pytest\nfrom utgen.raggraph.utils import normalize_signature", "code": "def test_normalize_signature_with_basic_function():\n    # Create a simple FunctionDef node\n    node = ast.FunctionDef(\n        name='example',\n        args=ast.arguments(\n            posonlyargs=[],\n            args=[],\n            vararg=None,\n            kwonlyargs=[],\n            kw_defaults=[],\n            kwarg=None,\n            defaults=[]\n        ),\n        body=[],\n        decorator_list=[],\n        returns=None,\n        type_comment=None\n    )\n    \n    result = normalize_signature(node)\n    assert result == 'def example()'"}, "test_normalize_signature_with_arguments": {"name": "test_normalize_signature_with_arguments", "imports": "import ast\nimport pytest\nfrom utgen.raggraph.utils import normalize_signature", "code": "def test_normalize_sig

In [10]:
# Split into lines, slice from index 1 to -1, then rejoin
lines = response.raw.splitlines()
test_content = "\n".join(lines[1:-1])

print(test_content)

In [11]:
test_content = response.raw

In [12]:
import json

# Convert string to dictionary
data = json.loads(test_content)

# Save to a file
with open("../data/tests_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)